## Sentinel-3A

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from xgboost.callback import EarlyStopping
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import timedelta

### Load and preprocess the TLE data


In [2]:
import pandas as pd

# Replace with appropriate path 
df_tles = pd.read_csv('/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/Sentinel-3A.csv', 
                      index_col=0, parse_dates=True)

#Check if datetime index is timezone-naive or timezone-aware
if df_tles.index.tz is None:
    df_tles.index = df_tles.index.tz_localize('UTC')
else:
    df_tles.index = df_tles.index.tz_convert('UTC')


print(df_tles.describe())
print(df_tles.index.inferred_type)



       eccentricity  argument of perigee  inclination  mean anomaly  \
count   2385.000000          2385.000000  2385.000000   2385.000000   
mean       0.000117             1.695867     1.721344     -1.693568   
std        0.000018             0.160225     0.000073      0.160251   
min        0.000067             0.300010     1.721205     -2.025364   
25%        0.000104             1.580202     1.721286     -1.815631   
50%        0.000117             1.703319     1.721334     -1.700993   
75%        0.000130             1.817951     1.721403     -1.577912   
max        0.000252             2.027614     1.721502     -0.297628   

       Brouwer mean motion  right ascension  
count         2.385000e+03      2385.000000  
mean          6.229033e-02         3.228459  
std           1.042236e-07         1.779778  
min           6.229006e-02         0.000230  
25%           6.229025e-02         1.722476  
50%           6.229034e-02         3.282257  
75%           6.229041e-02         4.7

### Extract and scale Brouwer mean motion

In [3]:
df_element_1 = df_tles[["Brouwer mean motion"]]
df_element_1 = (df_element_1 - df_element_1.mean())*1e7
df_element_1.describe()


,Brouwer mean motion
count,2.385000e+03
mean,2.304236e-11
std,1.042236e+00
min,-2.657132e+00
25%,-7.946519e-01
50%,9.919124e-02
75%,7.957418e-01
max,4.673058e+00


### Visualize the scaled element

In [4]:
import plotly.graph_objects as go


fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_element_1.index, 
    y=df_element_1[df_element_1.columns[0]],  
    mode="lines",  
    name="Orbital Element"
))

fig.update_layout(
    title="Brouwer mean motion Over Time",
    xaxis_title="Time",
    yaxis_title="Brouwer mean motion",
    template="plotly",
    width=1400
)

fig.show()


### Create lag features for time series forecasting

In [5]:
NUM_LAG_FEATURES = 3

df_y = df_element_1.copy()
df_x = df_element_1.shift(1).rename(columns={"Brouwer mean motion": "bmm_lag_1"})

for lag in range(2, NUM_LAG_FEATURES + 1):
    df_x[f"bmm_lag_{lag}"] = df_element_1.shift(lag)

# Drop rows with NaNs
df_x = df_x.iloc[NUM_LAG_FEATURES:]
df_y = df_y.iloc[NUM_LAG_FEATURES:]


### Split for hyperparameter tuning


In [6]:
# Split for tuning
split_index = int(len(df_x) * 0.8)
split_date = df_x.index[split_index].strftime("%Y-%m-%d")

df_x_train = df_x[:split_date]
df_y_train = df_y[:split_date]
df_x_test = df_x[split_date:]
df_y_test = df_y[split_date:]

# Tune XGBoost model with early stopping
tuned_model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=10,
    random_state=42
)

tuned_model.fit(
    df_x_train,
    df_y_train.values.ravel(),
    eval_set=[(df_x_train, df_y_train), (df_x_test, df_y_test)],
    verbose=True
)

# View best iteration and RMSE
best_n_estimators = tuned_model.best_iteration + 1  
print(f"Best number of trees: {best_n_estimators}")


[0]	validation_0-rmse:0.91928	validation_1-rmse:1.08310
[1]	validation_0-rmse:0.84117	validation_1-rmse:1.01113
[2]	validation_0-rmse:0.77170	validation_1-rmse:0.94770
[3]	validation_0-rmse:0.71030	validation_1-rmse:0.89299
[4]	validation_0-rmse:0.65601	validation_1-rmse:0.84637
[5]	validation_0-rmse:0.60814	validation_1-rmse:0.80632
[6]	validation_0-rmse:0.56621	validation_1-rmse:0.77146
[7]	validation_0-rmse:0.52935	validation_1-rmse:0.74120
[8]	validation_0-rmse:0.49741	validation_1-rmse:0.71544
[9]	validation_0-rmse:0.46985	validation_1-rmse:0.69393
[10]	validation_0-rmse:0.44612	validation_1-rmse:0.67598
[11]	validation_0-rmse:0.42576	validation_1-rmse:0.66095
[12]	validation_0-rmse:0.40849	validation_1-rmse:0.64858
[13]	validation_0-rmse:0.39377	validation_1-rmse:0.63772
[14]	validation_0-rmse:0.38125	validation_1-rmse:0.62937
[15]	validation_0-rmse:0.37075	validation_1-rmse:0.62264
[16]	validation_0-rmse:0.36192	validation_1-rmse:0.61654
[17]	validation_0-rmse:0.35452	validation


### Retrain final model on full dataset

In [7]:
# Use full data for final model
df_x_full = df_x.copy()
df_y_full = df_y.copy()

final_model = XGBRegressor(
    n_estimators=best_n_estimators,
    max_depth=3,
    learning_rate=0.1,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42
)

final_model.fit(df_x_full, df_y_full.values.ravel())


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=42,
             n_jobs=None, num_parallel_tree=None, ...)

### Make predictions on full data and compute residuals

In [8]:
# Predict and calculate residuals
y_pred_full = final_model.predict(df_x_full)
residuals_full = y_pred_full - df_y_full["Brouwer mean motion"].values

df_result = df_y_full.copy()
df_result["predicted"] = y_pred_full
df_result["residuals"] = residuals_full


###  Plot Observed vs Predicted (Full Time Range)

In [9]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["Brouwer mean motion"],
    mode='lines',
    name='Observed'
))

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["predicted"],
    mode='lines',
    name='Predicted'
))

fig.update_layout(
    title='Observed vs Predicted Brouwer Mean Motion (Full Series)',
    xaxis_title='Time',
    yaxis_title='Brouwer Mean Motion (scaled)',
    template='plotly_white',
    width=1200
)

fig.show()


### Plot residuals over time 

In [11]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals'
))

fig.add_hline(y=0, line_dash="dot", line_color="black")

fig.update_layout(
    title='Residuals Over Time',
    xaxis_title='Time',
    yaxis_title='Residual (Observed - Predicted)',
    template='plotly_white',
    width= 1400
)

fig.show()


### Plot Residuals with Ground Truth Maneuvers

In [12]:
# Load maneuver data
ground_truth_df = pd.read_csv(
    "/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/cleaned maneuver file/cleaned_SENTINEL-3A.csv",
    parse_dates=["Start_Timestamp", "End_Timestamp"]
)

for col in ["Start_Timestamp", "End_Timestamp"]:
    if ground_truth_df[col].dt.tz is None:
        ground_truth_df[col] = ground_truth_df[col].dt.tz_localize("UTC")
    else:
        ground_truth_df[col] = ground_truth_df[col].dt.tz_convert("UTC")


# Filter to full range
test_start = df_result.index.min()
test_end = df_result.index.max()

ground_truth_df_test = ground_truth_df[
    (ground_truth_df["Start_Timestamp"] >= test_start) &
    (ground_truth_df["Start_Timestamp"] <= test_end)
]

# Get residual range
y_min = df_result["residuals"].min()
y_max = df_result["residuals"].max()

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}'
))

fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left"
)

# Ground truth maneuver lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    hover_text = f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[y_min, y_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[hover_text, hover_text],
        showlegend=False
    ))

fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))

fig.update_layout(
    title='Residuals with Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Residual (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=600,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()

#### Detect Anomalies (Using 3σ Rule)

In [13]:

# Calculate mean and standard deviation of residuals
residual_mean = df_result["residuals"].mean()
residual_std = df_result["residuals"].std()
# Define upper and lower thresholds
upper = residual_mean + 3 * residual_std
lower = residual_mean - 3 * residual_std
# Flag anomalies: residuals that fall outside the threshold range
df_result["anomaly"] = (df_result["residuals"] > upper) | (df_result["residuals"] < lower)
print(f"Anomaly threshold range: {lower:.4f} to {upper:.4f}")
print(" Anomaly timestamps:")
print(df_result[df_result["anomaly"]].index)

Anomaly threshold range: -1.0969 to 1.0967
 Anomaly timestamps:
DatetimeIndex(['2016-07-22 04:45:05.575680+00:00',
               '2016-09-01 03:41:28.252607+00:00',
               '2016-11-02 03:34:02.368128+00:00',
               '2016-12-15 03:19:01.216128+00:00',
               '2017-03-15 21:16:15.042144+00:00',
               '2017-04-27 22:42:22.430015+00:00',
               '2017-07-13 14:21:12.840192+00:00',
               '2017-09-07 11:47:45.469535+00:00',
               '2017-12-14 04:22:41.753279+00:00',
               '2018-03-15 05:03:52.221311+00:00',
               '2018-05-25 04:22:49.531007+00:00',
               '2018-08-30 04:07:46.538688+00:00',
               '2018-12-20 14:09:58.006943+00:00',
               '2019-03-14 04:26:31.200576+00:00',
               '2019-06-14 03:00:37.035359+00:00',
               '2019-08-29 03:30:27.052416+00:00',
               '2019-12-12 03:07:58.987775+00:00',
               '2020-03-13 18:31:52.796639+00:00',
               '20

#### Plotting detected anomalies vs ground truth

In [14]:

# Load ground truth maneuver data
ground_truth_df = pd.read_csv("/Users/kurinjiarivazhagan/Desktop/trimester 5/satellite_data/orbital_elements/cleaned maneuver file/cleaned_SENTINEL-3A.csv",
    parse_dates=["Start_Timestamp", "End_Timestamp"]
)
# for non- fengyun satellites
for col in ["Start_Timestamp", "End_Timestamp"]:
    if ground_truth_df[col].dt.tz is None:
        ground_truth_df[col] = ground_truth_df[col].dt.tz_localize("UTC")
    else:
        ground_truth_df[col] = ground_truth_df[col].dt.tz_convert("UTC")

# Filter maneuver data to match df_result range 
test_start = df_result.index.min()
test_end = df_result.index.max()

ground_truth_df_test = ground_truth_df[
    (ground_truth_df["Start_Timestamp"] >= test_start) &
    (ground_truth_df["Start_Timestamp"] <= test_end)
]

# Get residual y-axis range for drawing vertical maneuver lines 
residuals_min = df_result["residuals"].min()
residuals_max = df_result["residuals"].max()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["Brouwer mean motion"],
    mode='lines+markers',
    name='Observed',
    marker=dict(size=4),
), secondary_y=False)

# Plot predicted values 
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["predicted"],
    mode='lines+markers',
    name='Predicted',
    marker=dict(size=4),
), secondary_y=False)

# Plot residuals 
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
), secondary_y=True)

# Plot detected anomalies 
fig.add_trace(go.Scatter(
    x=df_result[df_result["anomaly"]].index,
    y=df_result[df_result["anomaly"]]["residuals"],
    mode='markers',
    name='Detected Anomalies',
    marker=dict(color='red', size=10, symbol='circle'),
    hovertemplate='Anomaly Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}',
), secondary_y=True)

# Plot ground truth maneuver lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[residuals_min, residuals_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"] * 2,
        showlegend=False
    ))
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left",
    secondary_y=True
)
fig.update_layout(
    title='Observed vs Predicted with Residuals, Detected Anomalies, and Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Brouwer Mean Motion (scaled)',
    yaxis2_title='Residuals (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=700,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    xaxis_range=[test_start, test_end]
)

fig.show()


### Residuals with detected anomaly and ground truth maneuver

In [15]:

#Residual range for drawing maneuver lines 
residuals_min = df_result["residuals"].min()
residuals_max = df_result["residuals"].max()

# Create figure 
fig = go.Figure()

# Residuals
fig.add_trace(go.Scatter(
    x=df_result.index,
    y=df_result["residuals"],
    mode='lines+markers',
    name='Residuals',
    marker=dict(size=4),
    line=dict(color='blue')
))

#  Detected Anomalies
fig.add_trace(go.Scatter(
    x=df_result[df_result["anomaly"]].index,
    y=df_result[df_result["anomaly"]]["residuals"],
    mode='markers',
    name='Detected Anomalies',
    marker=dict(color='red', size=10, symbol='circle'),
    hovertemplate='Anomaly Date: %{x|%Y-%m-%d}<br>Residual: %{y:.4f}'
))

# Ground Truth Maneuver Lines
for _, row in ground_truth_df_test.iterrows():
    maneuver_date = row["Start_Timestamp"]
    fig.add_trace(go.Scatter(
        x=[maneuver_date, maneuver_date],
        y=[residuals_min, residuals_max],
        mode='lines',
        line=dict(color='green', dash='dash'),
        name='Ground Truth Maneuver',
        hoverinfo='text',
        text=[f"Maneuver Date: {maneuver_date.strftime('%Y-%m-%d')}"] * 2,
        showlegend=False
    ))
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color='green', dash='dash'),
    name='Ground Truth Maneuver'
))
fig.add_hline(
    y=0,
    line_dash="dot",
    line_color="black",
    annotation_text="Zero Residual",
    annotation_position="top left"
)
fig.update_layout(
    title='Residuals with Detected Anomalies and Ground Truth Maneuvers',
    xaxis_title='Time',
    yaxis_title='Residuals (Observed - Predicted)',
    template='plotly_white',
    width=1400,
    height=600,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    xaxis_range=[df_result.index.min(), df_result.index.max()]
)

fig.show()
